In [15]:
import numpy as np
import pandas as pd
from scipy.stats import norm
import yfinance as yf

rng = np.random.default_rng(42)

In [5]:
tickers = ["MSFT", "AAPL"]

data = yf.download(
    tickers,
    start="2011-09-01",
    end="2016-09-02",
    auto_adjust=False,
    progress=False
)

prices = data["Close"][tickers]
log_returns = np.log(prices / prices.shift(1)).dropna()
print("Price observations:", len(prices))
print("Return observations:", len(log_returns))
print("Price range:", prices.index[0], "to", prices.index[-1])
print("Return range:", log_returns.index[0], "to", log_returns.index[-1])

Price observations: 1259
Return observations: 1258
Price range: 2011-09-01 00:00:00 to 2016-09-01 00:00:00
Return range: 2011-09-02 00:00:00 to 2016-09-01 00:00:00


In [6]:
alpha = 0.95
K = 10
portfolio_value = 1_000_000

lambda_ewma = 0.97
theta_ewma = 0.97
M = 50_000

cap_msft = 448.77
cap_aapl = 575.11

weights = np.array([
    cap_msft / (cap_msft + cap_aapl),
    cap_aapl / (cap_msft + cap_aapl)
])

dollar_positions = portfolio_value * weights

print("Weights:", weights)
print("Dollar positions:", dollar_positions)

Weights: [0.43830332 0.56169668]
Dollar positions: [438303.31679494 561696.68320506]


# 1

In [8]:
mu = np.zeros(2)
Sigma = np.zeros((2, 2))

for x in log_returns.to_numpy():

    diff = x - mu
    Sigma = (
        theta_ewma * Sigma
        + (1 - theta_ewma) * np.outer(diff, diff)
    )
    mu = (
        lambda_ewma * mu
        + (1 - lambda_ewma) * x
    )

print("EWMA mean:")
print(mu)

print("EWMA covariance:")
print(Sigma)

EWMA mean:
[0.00125025 0.00106069]
EWMA covariance:
[[1.29897109e-04 3.30066088e-05]
 [3.30066088e-05 1.51851715e-04]]


# 2

In [9]:
theta_t = dollar_positions

loss_mean = -theta_t @ mu
loss_var = theta_t @ Sigma @ theta_t
loss_std = np.sqrt(loss_var)

z_alpha = norm.ppf(alpha)
var_1day = loss_mean + z_alpha * loss_std

var_Kday = np.sqrt(K) * var_1day

reg_capital = 3 * var_Kday

print("1-day loss mean:", loss_mean)
print("1-day loss std:", loss_std)
print("1-day VaR:", var_1day)
print("10-day VaR:", var_Kday)
print("Regulatory capital:", reg_capital)

1-day loss mean: -1143.7757727370135
1-day loss std: 9440.139311321771
1-day VaR: 14383.871612417772
10-day VaR: 45485.79586667885
Regulatory capital: 136457.38760003654


# 3

In [ ]:
mu_initial = mu.copy()
Sigma_initial = Sigma.copy()

sigma1 = np.sqrt(Sigma_initial[0, 0])
sigma2 = np.sqrt(Sigma_initial[1, 1])

rho = (
    Sigma_initial[0, 1]
    / (sigma1 * sigma2)
)

# (i) Apple -5 sigma shock
x2 = mu_initial[1] - 5 * sigma2

# (ii) Conditional distribution of Microsoft
mu1_cond = (
    mu_initial[0]
    + rho * sigma1 / sigma2
      * (x2 - mu_initial[1])
)

sigma1_cond = sigma1 * np.sqrt(1 - rho**2)
x1 = mu1_cond - 5 * sigma1_cond

shock = np.array([x1, x2])

In [14]:
# (iii) Update EWMA after the shock

diff = shock - mu_initial

Sigma_after_shock = (
    theta_ewma * Sigma_initial
    + (1 - theta_ewma)
      * np.outer(diff, diff)
)

mu_after_shock = (
    lambda_ewma * mu_initial
    + (1 - lambda_ewma) * shock
)

print("Mean after shock:")
print(mu_after_shock)

print("Covariance after shock:")
print(Sigma_after_shock)


Mean after shock:
[-0.00081323 -0.00078773]
Covariance after shock:
[[0.00026793 0.00015916]
 [0.00015916 0.00026118]]


# 4

In [21]:
path_return_sums = np.zeros((M, 2))

for m in range(M):

    mu_path = mu_after_shock.copy()
    Sigma_path = Sigma_after_shock.copy()

    sum_returns = shock.copy()

    for k in range(2, K + 1):

        x = rng.multivariate_normal(
            mu_path,
            Sigma_path
        )

        sum_returns += x
        if k < K:
            diff = x - mu_path

            Sigma_path = (
                theta_ewma * Sigma_path
                + (1 - theta_ewma)
                  * np.outer(diff, diff)
            )

            mu_path = (
                lambda_ewma * mu_path
                + (1 - lambda_ewma) * x
            )

    path_return_sums[m] = sum_returns

path_return_sums.shape

(50000, 2)

# 5

In [22]:
losses = -(path_return_sums @ dollar_positions)

print("Number of simulated losses:", len(losses))
print("First 5 losses:", losses[:5])

Number of simulated losses: 50000
First 5 losses: [88575.98201793 83514.38892628 81530.18746094  6648.98984099
 89296.95764393]


In [25]:
# (a) 
avg_loss = np.mean(losses)

# (b)
sorted_losses = np.sort(losses)
var_stressed = sorted_losses[int(np.ceil(M * alpha)) - 1]

# (c)
freq_var_exceed = np.mean(losses > var_Kday) * 100

# (d)
freq_capital_exceed = np.mean(losses > reg_capital) * 100

print("(a). Average 10-day loss:", avg_loss)
print("(b). Stressed 10-day VaR:", var_stressed)
print("(c). Exceed initial 10-day VaR (%):", freq_var_exceed)
print("(d). Exceed regulatory capital (%):", freq_capital_exceed)

(a). Average 10-day loss: 71089.51355772787
(b). Stressed 10-day VaR: 151293.81721925864
(c). Exceed initial 10-day VaR (%): 70.236
(d). Exceed regulatory capital (%): 8.872
